# Predict Customer Churn
## Score: .91339

In [10]:
N_FOLDS = 5
N_SEEDS = 3

In [11]:
import numpy as np
import pandas as pd

train = pd.read_csv("playground-series-s6e3/train.csv")
test = pd.read_csv("playground-series-s6e3/test.csv")
original = pd.read_csv("original_data/WA_Fn-UseC_-Telco-Customer-Churn.csv")
original = original.drop(columns=['customerID'])
original = original[train.columns.drop('id')]
train_combined = pd.concat([train.drop(columns=['id']), original], ignore_index=True)
sample_weights = np.array([1.0] * len(train) + [0.5] * len(original))
y_full = train_combined['Churn'].map({'Yes': 1, 'No': 0})
X_full = train_combined.drop(columns=['Churn'])
test_ids = test['id']
X_test_raw = test.drop(columns=['id'])
print(f"Train: {len(train)}, Original: {len(original)}, Combined: {len(X_full)}")

Train: 594194, Original: 7043, Combined: 601237


In [12]:
def engineer_features(df, total_charges_median=None):
    df = df.copy()
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
    median_tc = total_charges_median if total_charges_median is not None else df['TotalCharges'].median()
    df['TotalCharges'] = df['TotalCharges'].fillna(median_tc)
    df['AvgChargesPerMonth'] = df['TotalCharges'] / df['tenure'].replace(0, 1)
    df['MonthlyChargesSqrt'] = np.sqrt(df['MonthlyCharges'])
    df['tenure_squared'] = df['tenure'] ** 2
    df['ChargesRatio'] = df['TotalCharges'] / (df['MonthlyCharges'] * (df['tenure'] + 1))
    df['Contract_MonthToMonth'] = (df['Contract'] == 'Month-to-month').astype(int)
    df['Contract_OneYear'] = (df['Contract'] == 'One year').astype(int)
    df['Contract_TwoYear'] = (df['Contract'] == 'Two year').astype(int)
    df['IsFiberOptic'] = (df['InternetService'] == 'Fiber optic').astype(int)
    df['NumStreaming'] = ((df['StreamingTV'] == 'Yes') | (df['StreamingMovies'] == 'Yes')).astype(int)
    df['NumStreamingBoth'] = ((df['StreamingTV'] == 'Yes') & (df['StreamingMovies'] == 'Yes')).astype(int)
    addon_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport']
    df['NumAddons'] = sum((df[c] == 'Yes').astype(int) for c in addon_cols)
    df['PaymentElectronic'] = (df['PaymentMethod'] == 'Electronic check').astype(int)
    df['HasDependents'] = (df['Dependents'] == 'Yes').astype(int)
    df['HasPartner'] = (df['Partner'] == 'Yes').astype(int)
    df['SeniorWithShortTenure'] = (df['SeniorCitizen'] == 1) & (df['tenure'] < 12)
    df['SeniorWithShortTenure'] = df['SeniorWithShortTenure'].astype(int)
    df['HighChargeShortTenure'] = (df['MonthlyCharges'] > 70) & (df['tenure'] < 12)
    df['HighChargeShortTenure'] = df['HighChargeShortTenure'].astype(int)
    return df

def target_encode(train_df, test_df, col, target_series, m=5):
    global_mean = target_series.mean()
    combined = pd.DataFrame({col: train_df[col], '_y': target_series.values})
    agg = combined.groupby(col)['_y'].agg(['mean', 'count'])
    smooth = (agg['mean'] * agg['count'] + global_mean * m) / (agg['count'] + m)
    train_df = train_df.copy()
    test_df = test_df.copy()
    train_df[f'{col}_te'] = train_df[col].map(smooth).fillna(global_mean)
    test_df[f'{col}_te'] = test_df[col].map(smooth).fillna(global_mean)
    return train_df, test_df

X_full = engineer_features(X_full)
tc_median = X_full['TotalCharges'].median()
X_test_raw = engineer_features(X_test_raw, total_charges_median=tc_median)

for col in ['Contract', 'PaymentMethod', 'InternetService']:
    X_full, X_test_raw = target_encode(X_full, X_test_raw, col, y_full, m=20)
    X_full = X_full.drop(columns=[col])
    X_test_raw = X_test_raw.drop(columns=[col])

X_encoded = pd.get_dummies(X_full, drop_first=True)
X_test_encoded = pd.get_dummies(X_test_raw, drop_first=True)
X_test_encoded = X_test_encoded.reindex(columns=X_encoded.columns, fill_value=0)

In [13]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
fit_kw = {'sample_weight': sample_weights}

In [14]:
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV

xgb_grid = {
    'n_estimators': [300, 500],
    'max_depth': [4, 6],
    'learning_rate': [0.05, 0.1],
    'subsample': [0.8],
    'colsample_bytree': [0.8],
    'scale_pos_weight': [2],
}
xgb = XGBClassifier(random_state=42, eval_metric='logloss')
gs_xgb = GridSearchCV(xgb, xgb_grid, cv=cv, scoring='roc_auc', n_jobs=-1, verbose=1)
gs_xgb.fit(X_encoded, y_full, **fit_kw)
best_xgb = gs_xgb.best_estimator_
print(f"XGB Best CV AUC: {gs_xgb.best_score_:.4f}")

Fitting 5 folds for each of 8 candidates, totalling 40 fits
XGB Best CV AUC: 0.9160


In [15]:
from lightgbm import LGBMClassifier

lgb_grid = {
    'n_estimators': [300, 500],
    'max_depth': [4, 6],
    'learning_rate': [0.05, 0.1],
    'subsample': [0.8],
    'colsample_bytree': [0.8],
    'scale_pos_weight': [2],
}
lgb_base = LGBMClassifier(random_state=42, verbose=-1)
gs_lgb = GridSearchCV(lgb_base, lgb_grid, cv=cv, scoring='roc_auc', n_jobs=-1, verbose=1)
gs_lgb.fit(X_encoded, y_full, **fit_kw)
best_lgb = gs_lgb.best_estimator_
print(f"LGB Best CV AUC: {gs_lgb.best_score_:.4f}")

Fitting 5 folds for each of 8 candidates, totalling 40 fits
LGB Best CV AUC: 0.9159


In [16]:
from catboost import CatBoostClassifier

cat_grid = {
    'iterations': [300, 500],
    'depth': [4, 6],
    'learning_rate': [0.05, 0.1],
}
cat_base = CatBoostClassifier(random_state=42, verbose=0)
gs_cat = GridSearchCV(cat_base, cat_grid, cv=cv, scoring='roc_auc', n_jobs=-1, verbose=1)
gs_cat.fit(X_encoded, y_full, **fit_kw)
best_cat = gs_cat.best_estimator_
print(f"CatBoost Best CV AUC: {gs_cat.best_score_:.4f}")

Fitting 5 folds for each of 8 candidates, totalling 40 fits
CatBoost Best CV AUC: 0.9160


In [17]:
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

n_models = 5
oof = np.zeros((len(X_encoded), n_models))
test_preds = np.zeros((len(X_test_encoded), n_models))

def get_models(seed, fold):
    return [
        XGBClassifier(**gs_xgb.best_params_).set_params(random_state=seed+fold, eval_metric='logloss'),
        LGBMClassifier(**gs_lgb.best_params_).set_params(random_state=seed+fold, verbose=-1),
        CatBoostClassifier(iterations=gs_cat.best_params_['iterations'], depth=gs_cat.best_params_['depth'], learning_rate=gs_cat.best_params_['learning_rate'], random_state=seed+fold, verbose=0),
        RandomForestClassifier(n_estimators=300, max_depth=12, class_weight='balanced', random_state=seed+fold),
        ExtraTreesClassifier(n_estimators=300, max_depth=12, class_weight='balanced', random_state=seed+fold),
    ]

for fold, (train_idx, val_idx) in enumerate(cv.split(X_encoded, y_full)):
    X_tr, X_val = X_encoded.iloc[train_idx], X_encoded.iloc[val_idx]
    y_tr = y_full.iloc[train_idx]
    sw_tr = sample_weights[train_idx]
    models = get_models(42, fold)
    for i, m in enumerate(models):
        m.fit(X_tr, y_tr, sample_weight=sw_tr)
        oof[val_idx, i] = m.predict_proba(X_val)[:, 1]
        test_preds[:, i] += m.predict_proba(X_test_encoded)[:, 1] / N_FOLDS

meta = LogisticRegression(C=0.1, max_iter=1000, random_state=42)
meta.fit(oof, y_full)
stack_oof = meta.predict_proba(oof)[:, 1]
print(f"Stacking OOF AUC: {roc_auc_score(y_full, stack_oof):.4f}")

Stacking OOF AUC: 0.9153


In [18]:
all_test_preds = [test_preds.copy()]
for base_seed in [123, 456, 789][:N_SEEDS-1]:
    tp = np.zeros((len(X_test_encoded), n_models))
    for fold, (train_idx, val_idx) in enumerate(cv.split(X_encoded, y_full)):
        X_tr, X_val = X_encoded.iloc[train_idx], X_encoded.iloc[val_idx]
        y_tr = y_full.iloc[train_idx]
        sw_tr = sample_weights[train_idx]
        models = get_models(base_seed, fold)
        for i, m in enumerate(models):
            m.fit(X_tr, y_tr, sample_weight=sw_tr)
            tp[:, i] += m.predict_proba(X_test_encoded)[:, 1] / N_FOLDS
    all_test_preds.append(tp)

final_preds = np.mean([meta.predict_proba(tp)[:, 1] for tp in all_test_preds], axis=0)
submission = pd.DataFrame({'id': test_ids, 'Churn': final_preds})
submission.to_csv('submission.csv', index=False)
submission.head()

,id,Churn
0,594194,0.045961
1,594195,0.018183
2,594196,0.064153
3,594197,0.018884
4,594198,0.566702
